# People Counter Python SDK tutorial

This notebook teaches the stable `people_counter` Python API. It uses a bundled sample video and keeps model inference disabled until you explicitly opt in.

## 1. Prepare the environment

From the repository root, install exactly one hardware variant plus the experiment dependencies before starting Jupyter:

```bash
uv sync --extra cpu --extra experiments
uv run --extra cpu --extra experiments jupyter lab
```

Replace `cpu` with `gpu` only when the host has a compatible NVIDIA GPU and CUDA driver. The SDK deliberately does not fall back to another device.

In [ ]:
from pathlib import Path

from people_counter import (
    RFDetrBotsortConfig,
    RTDetrOsnetConfig,
    line_count_records,
    run,
    telemetry_records,
)


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the people-counter repository")


ROOT = find_repository_root(Path.cwd().resolve())
VIDEO = ROOT / "samples" / "three_people_walking.mp4"
RUN_INFERENCE = False
DEVICE_VARIANT = "cpu"
DEVICE = "cpu"

if not VIDEO.is_file():
    raise FileNotFoundError(f"Sample video not found: {VIDEO}")

print(f"Repository: {ROOT}")
print(f"Video: {VIDEO}")
print(f"Inference enabled: {RUN_INFERENCE}")

## 2. Construct a typed configuration

The directed line is expressed as source-video pixels: `(x1, y1, x2, y2)`. Reversing its endpoints swaps the meaning of `in` and `out`. Set `line=None` when line-crossing counts are not needed.

A config owns a mutable `RunResult` and is single-use. Build a fresh config for every invocation.

In [ ]:
def report_progress(current_result) -> None:
    total = current_result.total_sampled_frames
    print(f"Processed {current_result.processed_frames}/{total} sampled frames")


# Replace with (x1, y1, x2, y2) after checking the source dimensions.
COUNTING_LINE = None

config = RTDetrOsnetConfig(
    video=VIDEO,
    device_variant=DEVICE_VARIANT,
    device=DEVICE,
    batch_size=1,
    sample_fps=3.0,
    detection_threshold=0.6,
    use_fp16=False,
    line=COUNTING_LINE,
    progress_callback=report_progress,
)

config

## 3. Run the pipeline

Set `RUN_INFERENCE = True` in the setup cell and rerun the setup and configuration cells when you are ready. Model loading can download weights on the first run.

The `finally` block demonstrates durable partial-result handling. It does not catch or hide processing errors, so notebook and pipeline failures remain visible.

In [ ]:
result = None
persistable_telemetry = []
persistable_line_counts = []

if RUN_INFERENCE:
    try:
        result = run(config)
    finally:
        if config.result.initialized:
            persistable_telemetry = telemetry_records(config.result)
            persistable_line_counts = line_count_records(config.result)
else:
    print("Inference skipped. Set RUN_INFERENCE = True to process the sample.")

## 4. Inspect structured results

`RunResult` contains identity telemetry, optional per-frame line counts, throughput data, and early-termination state. The conversion helpers return typed built-in dictionaries and do not require pandas or Spark.

In [ ]:
if result is None:
    print("No result yet.")
else:
    summary = {
        "distinct_people": len(result.telemetry),
        "line_in_count": result.line_in_count,
        "line_out_count": result.line_out_count,
        "processed_frames": result.processed_frames,
        "effective_sample_fps": result.effective_sample_fps,
        "processing_seconds": result.processing_seconds,
        "ended_early": result.ended_early,
    }
    display(summary)
    display(persistable_telemetry[:5])
    display(persistable_line_counts[:5])

In [ ]:
if result is None:
    print("Run inference before creating DataFrames.")
else:
    import pandas as pd

    telemetry_frame = pd.DataFrame(persistable_telemetry)
    line_count_frame = pd.DataFrame(persistable_line_counts)
    display(telemetry_frame.head())
    display(line_count_frame.head())

## 5. Select the RF-DETR/BoT-SORT pipeline

The same top-level `run` function dispatches by config type. BoT-SORT has no long-lived appearance gallery, so counts can differ after long occlusions or re-entry.

In [ ]:
botsort_config = RFDetrBotsortConfig(
    video=VIDEO,
    device_variant=DEVICE_VARIANT,
    device=DEVICE,
    batch_size=1,
    sample_fps=3.0,
    detection_threshold=0.6,
    camera_motion_compensation=False,
)

if RUN_INFERENCE:
    botsort_result = run(botsort_config)
    print(f"BoT-SORT distinct people: {len(botsort_result.telemetry)}")
else:
    print("BoT-SORT inference skipped.")

## 6. Microsoft Fabric and OneLake

Build the wheel with `uv build`, upload it to a Fabric Environment, and attach that environment to the notebook. Use a Lakehouse path that OpenCV can open, such as `/lakehouse/default/Files/incoming/video.mp4`. If an `abfss://` URI cannot be opened directly, stage the video in notebook-local storage first.

Process one video's frames sequentially in one process. Scale out across videos with separate Fabric notebook activities. Add a stable run ID and use Delta merge semantics when activities can retry.

In [ ]:
RUN_FABRIC_EXAMPLE = False

if RUN_FABRIC_EXAMPLE:
    if result is None:
        raise RuntimeError("Run inference before writing Fabric tables")

    telemetry_rows = [
        {"source_video": VIDEO.name, **record}
        for record in telemetry_records(result)
    ]
    line_rows = [
        {"source_video": VIDEO.name, **record}
        for record in line_count_records(result)
    ]

    if telemetry_rows:
        spark.createDataFrame(telemetry_rows).write.format("delta").mode(
            "append"
        ).saveAsTable("people_counter_telemetry")

    if line_rows:
        spark.createDataFrame(line_rows).write.format("delta").mode(
            "append"
        ).saveAsTable("people_counter_line_counts")
else:
    print("Fabric write example skipped.")

## Next steps

- Enable a directed counting line after checking the source video's dimensions.
- Increase `sample_fps` for fast movement near the line.
- Use a fresh config for every video.
- Persist provenance such as source URI, run ID, model, thresholds, and processing timestamp with each record batch.